# Camada Bronze — Ingestão Landing → Bronze

**Projeto:** Stack Overgol  
**Arquitetura:** Medalhão (Landing → Bronze → Silver → Gold)  
**Responsabilidade desta camada:** Ingerir os arquivos CSV brutos da zona de Landing, enriquecer cada linha com metadados de rastreabilidade (origem, timestamp, hash) e persistir em formato Delta Lake, particionado por data de ingestão — sem nenhuma transformação de negócio.  

---

### Princípios da Camada Bronze
| Princípio | Aplicação neste notebook |
|---|---|
| **Fidelidade aos dados brutos** | Schema inferido; nenhuma coluna é modificada ou removida |
| **Rastreabilidade** | Colunas `_src_file`, `_ingested_at` e `_row_hash` adicionadas a cada linha |
| **Idempotência** | Modo `overwrite` com `overwriteSchema=True` garante reexecução segura |
| **Governança** | Tabelas registradas no Unity Catalog sob `stack_overgol.bronze` |
| **Observabilidade** | Log estruturado ao final registra resultado de cada tabela ingerida |

## 1. Configuração do Ambiente

Criação do **Catálogo** e dos três **Schemas** da arquitetura Medalhão no Unity Catalog.  
O uso de `IF NOT EXISTS` torna este bloco idempotente — pode ser re-executado sem erro.

In [0]:
%sql
-- Cria o catálogo principal do projeto (se ainda não existir)
CREATE CATALOG IF NOT EXISTS stack_overgol
  COMMENT 'Catálogo principal do projeto Stack Overgol';

USE CATALOG stack_overgol;

-- Cria os três schemas da arquitetura Medalhão
CREATE SCHEMA IF NOT EXISTS bronze
  COMMENT 'Dados brutos ingeridos da zona de Landing. Nenhuma transformação de negócio aplicada.';

CREATE SCHEMA IF NOT EXISTS silver
  COMMENT 'Dados limpos, tipados e validados. Regras de negócio aplicadas.';

CREATE SCHEMA IF NOT EXISTS gold
  COMMENT 'Dados agregados e modelados para consumo analítico.';

-- Cria o Volume de Landing dentro do schema default (área de stage para arquivos brutos)
CREATE VOLUME IF NOT EXISTS stack_overgol.default.landing
  COMMENT 'Volume de entrada para arquivos CSV brutos antes da ingestão na Bronze.';

## 2. Imports e Constantes Globais

Centralizamos aqui todas as dependências e constantes do pipeline.  
Isso evita "magic strings" espalhadas pelo código e facilita manutenção futura.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from datetime import datetime, timezone
from typing import Optional
import traceback

# ---------------------------------------------------------------------------
# Constantes do pipeline
# ---------------------------------------------------------------------------

# Caminho do Volume de Landing no Unity Catalog
LANDING_PATH: str = "/Volumes/stack_overgol/default/landing"

# Catálogo e schema de destino
TARGET_CATALOG: str = "stack_overgol"
TARGET_SCHEMA:  str = "bronze"

# Timestamp único de referência para toda a execução do pipeline
# Usar um único valor garante consistência entre todas as tabelas ingeridas na mesma run
PIPELINE_RUN_AT: datetime = datetime.now(timezone.utc)

print(f"Pipeline iniciado em: {PIPELINE_RUN_AT.isoformat()}")
print(f"Landing Path : {LANDING_PATH}")
print(f"Destino      : {TARGET_CATALOG}.{TARGET_SCHEMA}")

Pipeline iniciado em: 2026-05-05T02:07:01.796540+00:00
Landing Path : /Volumes/stack_overgol/default/landing
Destino      : stack_overgol.bronze


## 3. Configuração Declarativa por Tabela

Em vez de um loop genérico que trata todas as tabelas igualmente, definimos aqui um **registro de configuração** para cada tabela.  
Isso permite que o pipeline saiba:
- Qual coluna usar para **particionar** a tabela Delta (melhora performance de leitura)
- Qual é a **chave primária** lógica de cada entidade (documentação e uso futuro na Silver)
- Qual **separador** e **encoding** o CSV usa (evita erros silenciosos)

Adicionar uma nova tabela ao pipeline = adicionar um dicionário nesta lista.

In [0]:
# ---------------------------------------------------------------------------
# Registro de configuração das tabelas
# Chaves:
#   file_name       : nome exato do arquivo CSV no Volume de Landing
#   table_name      : nome da tabela Delta que será criada na Bronze
#   primary_key     : chave primária lógica (documentação; não enforced na Bronze)
#   partition_col   : coluna de data usada para particionar a tabela Delta
#                     None → particiona pela coluna de metadado _ingestion_date
#   sep             : delimitador do CSV
#   encoding        : encoding do arquivo
# ---------------------------------------------------------------------------

TABLE_CONFIG: list[dict] = [
    {
        "file_name"    : "clientes.csv",
        "table_name"   : "clientes",
        "primary_key"  : "id_cliente",
        "partition_col": None,   # sem coluna de data de negócio relevante para partição
        "sep"          : ",",
        "encoding"     : "UTF-8",
    },
    {
        "file_name"    : "pedidos.csv",
        "table_name"   : "pedidos",
        "primary_key"  : "id_pedido",
        "partition_col": "data_pedido",   # particiona por data do pedido
        "sep"          : ",",
        "encoding"     : "UTF-8",
    },
    {
        "file_name"    : "avaliacoes.csv",
        "table_name"   : "avaliacoes",
        "primary_key"  : "id_avaliacao",
        "partition_col": "data_avaliacao",
        "sep"          : ",",
        "encoding"     : "UTF-8",
    },
    {
        "file_name"    : "catalogo_produtos.csv",
        "table_name"   : "catalogo_produtos",
        "primary_key"  : "id_produto",
        "partition_col": None,
        "sep"          : ",",
        "encoding"     : "UTF-8",
    },
    {
        "file_name"    : "clickstream.csv",
        "table_name"   : "clickstream",
        "primary_key"  : "id_evento",
        "partition_col": "data_evento",   # alto volume; partição por data é essencial
        "sep"          : ",",
        "encoding"     : "UTF-8",
    },
    {
        "file_name"    : "suporte_tickets.csv",
        "table_name"   : "suporte_tickets",
        "primary_key"  : "ticket_id",
        "partition_col": "data_abertura",
        "sep"          : ",",
        "encoding"     : "UTF-8",
    },
]

print(f"{len(TABLE_CONFIG)} tabelas registradas para ingestão:")
for cfg in TABLE_CONFIG:
    print(f"  • {cfg['file_name']:30s} → {TARGET_SCHEMA}.{cfg['table_name']}")

6 tabelas registradas para ingestão:
  • clientes.csv                   → bronze.clientes
  • pedidos.csv                    → bronze.pedidos
  • avaliacoes.csv                 → bronze.avaliacoes
  • catalogo_produtos.csv          → bronze.catalogo_produtos
  • clickstream.csv                → bronze.clickstream
  • suporte_tickets.csv            → bronze.suporte_tickets


## 4. Funções Auxiliares

Encapsulamos toda a lógica do pipeline em funções com responsabilidade única.  
Isso torna o código **testável**, **legível** e **reutilizável** em outros notebooks.

### Funções definidas:
| Função | Responsabilidade |
|---|---|
| `sanitize_column_names` | Remove/substitui caracteres especiais nos nomes de colunas para compatibilidade Delta |
| `read_csv_from_landing` | Lê o CSV com as opções certas e retorna um DataFrame Spark |
| `add_ingestion_metadata` | Adiciona colunas de metadados de rastreabilidade a cada linha |
| `write_bronze_table` | Persiste o DataFrame como tabela Delta com as opções corretas |
| `ingest_table` | Orquestra as quatro funções acima para uma tabela; captura erros |

> **Nota sobre `sanitize_column_names`:** O Delta Lake rejeita colunas com caracteres especiais  
> (espaços, `.`, `,`, `;`, `{}`, `()`, `=`, etc.).

In [0]:
import re

def sanitize_column_names(df: DataFrame) -> DataFrame:
    """
    Renomeia colunas do DataFrame para garantir compatibilidade com o Delta Lake.

    O Delta Lake rejeita colunas cujos nomes contenham certos caracteres especiais.

    Caracteres tratados:
      - Espaços                → underscore (_)
      - Ponto (.)              → underscore (_)  ex: 'preco.unitario' → 'preco_unitario'
      - Vírgula, ponto-e-vírgula, dois-pontos → removidos
      - Chaves { }             → removidos
      - Parênteses ( )         → removidos
      - Sinal de igual (=)     → removido
      - Qualquer outro char não alfanumérico além de _ → underscore
      - Múltiplos underscores consecutivos → colapsados em um único
      - Underscores no início/fim → removidos

    Além da renomeação, imprime no log um aviso para cada coluna que foi alterada,
    garantindo rastreabilidade da transformação.

    Args:
        df : DataFrame com possíveis nomes de colunas problemáticos

    Returns:
        DataFrame com todos os nomes de colunas compatíveis com Delta Lake
    """
    renamed = []
    for col in df.columns:
        # Substitui qualquer caractere não alfanumérico (exceto _) por underscore
        clean = re.sub(r'[^\w]', '_', col)
        # Colapsa múltiplos underscores consecutivos
        clean = re.sub(r'_+', '_', clean)
        # Remove underscores no início e no fim
        clean = clean.strip('_')
        # Garante que o nome não fique vazio (fallback)
        if not clean:
            clean = f'col_{df.columns.index(col)}'
        renamed.append((col, clean))

    # Aplica renomeações e loga apenas as colunas que foram alteradas
    for original, sanitized in renamed:
        if original != sanitized:
            print(f"    [sanitize] '{original}' → '{sanitized}'")
            df = df.withColumnRenamed(original, sanitized)

    return df


def read_csv_from_landing(
    file_path: str,
    sep: str = ",",
    encoding: str = "UTF-8",
) -> DataFrame:
    """
    Lê um arquivo CSV do Volume de Landing e retorna um DataFrame Spark.

    Opções relevantes:
      - header=True        : primeira linha é o cabeçalho
      - inferSchema=True   : Spark infere os tipos de cada coluna
      - multiLine=True     : suporta campos de texto com quebras de linha (ex: coluna 'comentario')
      - escape='"'         : trata aspas duplas escapadas dentro de campos
      - mode=PERMISSIVE    : linhas malformadas são mantidas com campos nulos
                             (preferível ao DROPMALFORMED na Bronze, que descartaria dados brutos)

    Args:
        file_path : caminho completo do arquivo no Volume
        sep       : delimitador de campos (padrão: vírgula)
        encoding  : encoding do arquivo (padrão: UTF-8)

    Returns:
        DataFrame Spark com schema inferido
    """
    return (
        spark.read
        .option("header",      True)
        .option("inferSchema", True)
        .option("sep",         sep)
        .option("encoding",    encoding)
        .option("multiLine",   True)
        .option("escape",      '"')
        .option("mode",        "PERMISSIVE")
        .csv(file_path)
    )


def add_ingestion_metadata(
    df: DataFrame,
    file_path: str,
    ingested_at: datetime,
) -> DataFrame:
    """
    Adiciona colunas de metadados de rastreabilidade a cada linha do DataFrame.

    Colunas adicionadas (prefixo '_' indica metadado de pipeline, não dado de negócio):
      _src_file      : caminho completo do arquivo CSV de origem
      _ingested_at   : timestamp UTC do momento da ingestão (consistente para toda a run)
      _ingestion_date: data de ingestão (coluna de partição padrão)
      _row_hash      : hash MD5 de todas as colunas de negócio concatenadas.
                       Serve para detectar alterações em reprocessamentos futuros
                       e como chave de deduplicação na Silver.

    Args:
        df          : DataFrame sem metadados
        file_path   : caminho do arquivo de origem para rastreabilidade
        ingested_at : timestamp da execução do pipeline

    Returns:
        DataFrame enriquecido com as quatro colunas de metadado
    """
    # Concatena todas as colunas de negócio em uma única string para gerar o hash.
    # F.concat_ws usa '|' como separador para evitar colisões (ex: ['a','b'] ≠ ['ab',''])
    all_cols = [F.col(c).cast("string") for c in df.columns]

    return (
        df
        .withColumn("_src_file",       F.lit(file_path))
        .withColumn("_ingested_at",    F.lit(ingested_at.isoformat()).cast("timestamp"))
        .withColumn("_ingestion_date", F.lit(ingested_at.date().isoformat()).cast("date"))
        .withColumn("_row_hash",       F.md5(F.concat_ws("|", *all_cols)))
    )


def write_bronze_table(
    df: DataFrame,
    table_full_name: str,
    partition_col: Optional[str],
) -> None:
    """
    Persiste o DataFrame como tabela Delta no schema Bronze.

    Opções de escrita:
      - format=delta         : formato colunar com suporte a ACID e time travel
      - mode=overwrite       : idempotente — reexecutar o pipeline não gera duplicatas
      - overwriteSchema=True : permite que mudanças de schema no CSV sejam propagadas
                               sem necessidade de dropar a tabela manualmente

    Particionamento:
      Sempre usamos _ingestion_date (coluna de metadado, tipo DATE, gerada pelo pipeline).
      Não tentamos converter as colunas de data de negócio (ex: data_pedido) porque
      os dados brutos têm formatos inválidos (ex: '2025/26/02') que causam
      CAST_INVALID_INPUT ao escrever no Delta. Padronização de datas é
      responsabilidade da Silver.

    Args:
        df              : DataFrame com dados de negócio + metadados
        table_full_name : nome completo da tabela (catalog.schema.table)
        partition_col   : coluna de partição declarada (documentação; não usada na Bronze)
    """
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .partitionBy("_ingestion_date")
        .saveAsTable(table_full_name)
    )


def ingest_table(cfg: dict, run_at: datetime) -> dict:
    """
    Orquestra a ingestão completa de uma tabela: leitura → metadados → escrita.

    Captura qualquer exceção sem interromper o pipeline, registrando o erro
    no log de resultado para diagnóstico posterior.

    Args:
        cfg    : dicionário de configuração da tabela (ver TABLE_CONFIG)
        run_at : timestamp da execução do pipeline

    Returns:
        Dicionário com resultado da ingestão desta tabela:
          status       : 'SUCCESS' ou 'FAILED'
          table        : nome da tabela
          rows_ingested: número de linhas escritas (0 em caso de falha)
          error        : mensagem de erro (None em caso de sucesso)
    """
    file_path       = f"{LANDING_PATH}/{cfg['file_name']}"
    table_full_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{cfg['table_name']}"

    result = {
        "table"        : table_full_name,
        "status"       : None,
        "rows_ingested": 0,
        "error"        : None,
    }

    try:
        # Leitura do CSV bruto
        df = read_csv_from_landing(
            file_path = file_path,
            sep       = cfg["sep"],
            encoding  = cfg["encoding"],
        )

        # Sanitização dos nomes de colunas para compatibilidade Delta Lake
        df = sanitize_column_names(df)

        # Enriquecimento com metadados de rastreabilidade
        df = add_ingestion_metadata(
            df          = df,
            file_path   = file_path,
            ingested_at = run_at,
        )

        # Contagem de linhas (ação Spark — força a materialização do DataFrame)
        row_count = df.count()

        # Escrita Delta particionada
        write_bronze_table(
            df              = df,
            table_full_name = table_full_name,
            partition_col   = cfg["partition_col"],
        )

        result["status"]        = "SUCCESS"
        result["rows_ingested"] = row_count

    except Exception as e:
        # Registra o erro completo sem interromper o pipeline
        result["status"] = "FAILED"
        result["error"]  = traceback.format_exc()
        print(f"  [ERRO] {table_full_name}: {e}")

    return result


print("Funções auxiliares carregadas com sucesso.")

Funções auxiliares carregadas com sucesso.


## 5. Execução do Pipeline de Ingestão

Iteramos sobre o `TABLE_CONFIG` e chamamos `ingest_table` para cada tabela.  
O resultado de cada ingestão é acumulado em `ingestion_log` para o relatório final.

In [0]:
ingestion_log: list[dict] = []

print(f"{'='*65}")
print(f"  PIPELINE BRONZE — Início: {PIPELINE_RUN_AT.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"{'='*65}\n")

for i, cfg in enumerate(TABLE_CONFIG, start=1):
    table_label = f"{TARGET_SCHEMA}.{cfg['table_name']}"
    print(f"[{i}/{len(TABLE_CONFIG)}] Ingerindo: {table_label} ...")

    result = ingest_table(cfg, PIPELINE_RUN_AT)
    ingestion_log.append(result)

    if result["status"] == "SUCCESS":
        print(f"        ✓ {result['rows_ingested']:,} linhas escritas → {result['table']}\n")
    else:
        print(f"        ✗ FALHA — ver log abaixo\n")

pipeline_end = datetime.now(timezone.utc)
duration_s   = (pipeline_end - PIPELINE_RUN_AT).total_seconds()

print(f"{'='*65}")
print(f"  Pipeline finalizado em {duration_s:.1f}s")
print(f"{'='*65}")

  PIPELINE BRONZE — Início: 2026-05-05 02:07:01 UTC

[1/6] Ingerindo: bronze.clientes ...
        ✓ 61,345 linhas escritas → stack_overgol.bronze.clientes

[2/6] Ingerindo: bronze.pedidos ...
        ✓ 314,900 linhas escritas → stack_overgol.bronze.pedidos

[3/6] Ingerindo: bronze.avaliacoes ...
        ✓ 156,832 linhas escritas → stack_overgol.bronze.avaliacoes

[4/6] Ingerindo: bronze.catalogo_produtos ...
        ✓ 517 linhas escritas → stack_overgol.bronze.catalogo_produtos

[5/6] Ingerindo: bronze.clickstream ...
        ✓ 500,000 linhas escritas → stack_overgol.bronze.clickstream

[6/6] Ingerindo: bronze.suporte_tickets ...
        ✓ 34,697 linhas escritas → stack_overgol.bronze.suporte_tickets

  Pipeline finalizado em 53.0s


## 6. Relatório de Execução (Log Estruturado)

Exibe um resumo completo da execução em formato tabular.  
Em um ambiente de produção, este DataFrame seria escrito em uma tabela de auditoria
(ex: `stack_overgol.default.pipeline_audit_log`) para rastreabilidade histórica.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

# Monta o schema explícito do log para garantir tipos corretos
log_schema = StructType([
    StructField("table",         StringType(), nullable=False),
    StructField("status",        StringType(), nullable=False),
    StructField("rows_ingested", LongType(),   nullable=False),
    StructField("error",         StringType(), nullable=True),
])

log_rows = [
    (r["table"], r["status"], r["rows_ingested"], r["error"])
    for r in ingestion_log
]

df_log = spark.createDataFrame(log_rows, schema=log_schema)

# Exibe o relatório no notebook
print("\n📋 RELATÓRIO DE EXECUÇÃO — CAMADA BRONZE")
print(f"   Run timestamp : {PIPELINE_RUN_AT.isoformat()}")
print(f"   Duração total : {duration_s:.1f}s")
print(f"   Tabelas OK    : {sum(1 for r in ingestion_log if r['status'] == 'SUCCESS')}")
print(f"   Tabelas FAIL  : {sum(1 for r in ingestion_log if r['status'] == 'FAILED')}\n")

display(df_log)

# -------------------------------------------------------------------------
# Opcional (produção): salvar o log como tabela de auditoria - revisores da bronze oq vcs acham??
# -------------------------------------------------------------------------
# (
#     df_log
#     .withColumn("run_at", F.lit(PIPELINE_RUN_AT.isoformat()).cast("timestamp"))
#     .write.format("delta")
#     .mode("append")
#     .saveAsTable("stack_overgol.default.pipeline_audit_log")
# )

# Falha o notebook se qualquer tabela falhou (importante em pipelines orquestrados)
failed = [r for r in ingestion_log if r["status"] == "FAILED"]
if failed:
    failed_names = ", ".join(r["table"] for r in failed)
    raise RuntimeError(
        f"Pipeline Bronze finalizado com {len(failed)} falha(s): {failed_names}. "
        "Verifique o relatório acima para detalhes."
    )


📋 RELATÓRIO DE EXECUÇÃO — CAMADA BRONZE
   Run timestamp : 2026-05-05T02:07:01.796540+00:00
   Duração total : 53.0s
   Tabelas OK    : 6
   Tabelas FAIL  : 0



table,status,rows_ingested,error
stack_overgol.bronze.clientes,SUCCESS,61345,null
stack_overgol.bronze.pedidos,SUCCESS,314900,null
stack_overgol.bronze.avaliacoes,SUCCESS,156832,null
stack_overgol.bronze.catalogo_produtos,SUCCESS,517,null
stack_overgol.bronze.clickstream,SUCCESS,500000,null
stack_overgol.bronze.suporte_tickets,SUCCESS,34697,null


## 7. Verificação Pós-Ingestão

Consulta de sanidade rápida para confirmar que as tabelas foram criadas corretamente no Unity Catalog.  
Em produção, este bloco pode ser substituído por um teste de qualidade com **Great Expectations** ou **dbt tests**.

In [0]:
%sql
-- Lista todas as tabelas registradas no schema Bronze
SHOW TABLES IN stack_overgol.bronze;

database,tableName,isTemporary
bronze,avaliacoes,false
bronze,catalogo_produtos,false
bronze,clickstream,false
bronze,clientes,false
bronze,pedidos,false
bronze,suporte_tickets,false


In [0]:
%sql
-- Verifica contagem de linhas, partições e metadados das tabelas Bronze
-- Substitua o nome da tabela conforme necessário
SELECT
    _ingestion_date,
    _src_file,
    COUNT(*)        AS total_linhas,
    COUNT(DISTINCT _row_hash) AS linhas_unicas
FROM stack_overgol.bronze.pedidos
GROUP BY _ingestion_date, _src_file
ORDER BY _ingestion_date DESC;

_ingestion_date,_src_file,total_linhas,linhas_unicas
2026-05-05,/Volumes/stack_overgol/default/landing/pedidos.csv,314900,314900


---
## Resumo do que foi construído

| Componente | Decisão técnica | Justificativa |
|---|---|---|
| **Formato de armazenamento** | Delta Lake | ACID, time travel, schema evolution |
| **Idempotência** | `overwrite` + `overwriteSchema=True` | Reexecução segura sem duplicatas |
| **Rastreabilidade** | `_src_file`, `_ingested_at`, `_row_hash` | Auditoria e detecção de mudanças na Silver |
| **Particionamento** | Coluna de data de negócio (ou `_ingestion_date`) | Partition pruning em queries downstream |
| **Tratamento de erros** | Try/except por tabela + raise no final | Pipeline não para na primeira falha; orquestradores detectam falha |
| **Configuração** | Declarativa em `TABLE_CONFIG` | Adicionar nova tabela = 1 dicionário, sem tocar na lógica |
| **Schema** | Inferido (`inferSchema=True`) + `PERMISSIVE` | Bronze preserva dado bruto; transformações ficam na Silver |

### Próximos passos — Camada Silver
- Padronização de `status`, `metodo_pagamento`, `categoria`, `ativo`, `recomenda` e demais campos sujos
- Tipagem correta de datas, valores numéricos e booleanos
- Deduplicação usando `_row_hash`
- Validação de integridade referencial entre tabelas
- Correção do campo `estado` em `clientes` (cidades no lugar de estados)